# Experiment 8


In [ ]:
import os, re, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()
 
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_EVAL = 100 if DEVICE.type == "cpu" else 1000   # None = whole Hindi test split (8,847 articles)
LANGUAGE = "hindi"
print("Device:", DEVICE)

In [ ]:
def load_split(split):
    files = {"train": ["train/0000.parquet", "train/0001.parquet"],
             "validation": ["validation/0000.parquet"], "test": ["test/0000.parquet"]}[split]
    paths = [hf_hub_download("csebuetnlp/xlsum", f"{LANGUAGE}/{f}", repo_type="dataset",
                             revision="refs/convert/parquet") for f in files]
    return pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
 
test_df = load_split("test")
print("Hindi test split:", test_df.shape)
print(test_df.columns.tolist())
test_df[["title", "summary", "text"]].head(3)

In [ ]:
test_df["text_words"] = test_df["text"].str.split().str.len()
test_df["summary_words"] = test_df["summary"].str.split().str.len()
print(test_df[["text_words", "summary_words"]].describe().round(1))
 
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
test_df["text_words"].clip(upper=2000).plot.hist(bins=50, ax=ax[0], title="Article length (words)")
test_df["summary_words"].plot.hist(bins=40, ax=ax[1], title="Reference summary length (words)", color="tab:orange")
plt.tight_layout(); plt.show()

In [ ]:
eval_df = test_df if N_EVAL is None else test_df.sample(N_EVAL, random_state=SEED).reset_index(drop=True)
WHITESPACE_HANDLER = lambda k: re.sub(r"\s+", " ", re.sub(r"\n+", " ", k.strip()))
articles = [WHITESPACE_HANDLER(t) for t in eval_df["text"]]
references = [WHITESPACE_HANDLER(s) for s in eval_df["summary"]]
print("Evaluation articles:", len(articles))
print("\nSample article  :", articles[0][:400], "...")
print("Reference summary:", references[0])

In [ ]:
GEN_KWARGS = dict(num_beams=4, max_new_tokens=84, no_repeat_ngram_size=2, length_penalty=1.0, early_stopping=True)
MAX_INPUT = 512
BATCH = 16 if DEVICE.type == "cuda" else 4
 
def summarize(model, tokenizer, texts, extra_gen_kwargs=None, name=""):
    model.eval(); outputs, t0 = [], time.time()
    for i in range(0, len(texts), BATCH):
        enc = tokenizer(texts[i:i + BATCH], return_tensors="pt", truncation=True,
                        max_length=MAX_INPUT, padding=True).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(**enc, **GEN_KWARGS, **(extra_gen_kwargs or {}))
        outputs += tokenizer.batch_decode(ids, skip_special_tokens=True)
        if (i // BATCH + 1) % 5 == 0:
            print(f"[{name}] {len(outputs)}/{len(texts)} summaries | {time.time() - t0:.0f}s")
    elapsed = time.time() - t0
    print(f"[{name}] done: {len(outputs)} summaries in {elapsed:.0f}s ({elapsed / len(texts):.2f}s / article)")
    return [o.strip() for o in outputs], elapsed

In [ ]:
MT5_NAME = "csebuetnlp/mT5_multilingual_XLSum"
mt5_tok = AutoTokenizer.from_pretrained(MT5_NAME)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MT5_NAME).to(DEVICE)
print(f"mT5 parameters: {sum(p.numel() for p in mt5_model.parameters()) / 1e6:.0f}M")
 
mt5_summaries, mt5_time = summarize(mt5_model, mt5_tok, articles, name="mT5")
del mt5_model

In [ ]:
MBART_NAME = "mskayacioglu/mbart-xlsum-2"
mbart_tok = AutoTokenizer.from_pretrained(MBART_NAME)
mbart_tok.src_lang = "hi_IN"
mbart_tok.tgt_lang = "hi_IN"
mbart_model = AutoModelForSeq2SeqLM.from_pretrained(MBART_NAME).to(DEVICE)
HI_TOKEN_ID = mbart_tok.convert_tokens_to_ids("hi_IN")
print(f"mBART parameters: {sum(p.numel() for p in mbart_model.parameters()) / 1e6:.0f}M | hi_IN token id = {HI_TOKEN_ID}")
 
mbart_summaries, mbart_time = summarize(mbart_model, mbart_tok, articles,
                                        extra_gen_kwargs={"forced_bos_token_id": HI_TOKEN_ID}, name="mBART")
del mbart_model

In [ ]:
for k in range(3):
    print(f"================ Example {k + 1} ================")
    print("ARTICLE  :", articles[k][:350], "...")
    print("REFERENCE:", references[k])
    print("mT5      :", mt5_summaries[k])
    print("mBART    :", mbart_summaries[k], "\n")

In [ ]:
from rouge_score import rouge_scorer
 
class DevanagariTokenizer:
    def tokenize(self, text):
        text = re.sub(r"[\u0964\u0965]", " ", text)                 # danda / double danda
        text = re.sub(r"[^\w\s\u0900-\u097F]", " ", text.lower())   # punctuation
        return text.split()
 
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False, tokenizer=DevanagariTokenizer())
print("Tokenizer check:", DevanagariTokenizer().tokenize(references[0])[:12])
 
def rouge_scores(preds, refs):
    per = [scorer.score(r, p) for p, r in zip(preds, refs)]
    rows = {m: [s[m].fmeasure for s in per] for m in ("rouge1", "rouge2", "rougeL")}
    return pd.DataFrame(rows)
 
rouge_mt5, rouge_mbart = rouge_scores(mt5_summaries, references), rouge_scores(mbart_summaries, references)

In [ ]:
from bert_score import score as bert_score
 
def mbert_scores(preds, refs):
    # mBERT (bert-base-multilingual-cased), using layer 9 for BERTScore
    P, R, F = bert_score(preds, refs, model_type="google-bert/bert-base-multilingual-cased", num_layers=9,
                         batch_size=16, device=str(DEVICE), verbose=False)
    return pd.DataFrame({"P": P.numpy(), "R": R.numpy(), "F1": F.numpy()})
 
bs_mt5, bs_mbart = mbert_scores(mt5_summaries, references), mbert_scores(mbart_summaries, references)

In [ ]:
def summary_row(rouge_df, bs_df, preds, secs):
    return {
        "ROUGE-1": rouge_df["rouge1"].mean() * 100,
        "ROUGE-2": rouge_df["rouge2"].mean() * 100,
        "ROUGE-L": rouge_df["rougeL"].mean() * 100,
        "mBERTScore-P": bs_df["P"].mean() * 100,
        "mBERTScore-R": bs_df["R"].mean() * 100,
        "mBERTScore-F1": bs_df["F1"].mean() * 100,
        "Avg summary length (words)": np.mean([len(p.split()) for p in preds]),
        "Seconds / article": secs / len(preds),
    }
 
results = pd.DataFrame({
    "mT5": summary_row(rouge_mt5, bs_mt5, mt5_summaries, mt5_time),
    "mBART": summary_row(rouge_mbart, bs_mbart, mbart_summaries, mbart_time),
}).T.round(2)
print(f"Hindi XL-Sum – {len(articles)} test articles (F1 scores x 100)")
print("Average reference length (words):", round(np.mean([len(r.split()) for r in references]), 2))
results

In [ ]:
metrics = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "mBERTScore-F1"]
ax = results[metrics].T.plot.bar(figsize=(8, 4), rot=0, color=["tab:blue", "tab:orange"])
ax.set_ylabel("Score (x100)"); ax.set_title("mT5 vs mBART – Hindi XL-Sum")
for c in ax.containers:
    ax.bar_label(c, fmt="%.1f", fontsize=8)
plt.tight_layout(); plt.show()
 
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
pairs = [(rouge_mt5["rouge1"], rouge_mbart["rouge1"], "ROUGE-1"), (rouge_mt5["rouge2"], rouge_mbart["rouge2"], "ROUGE-2"),
         (rouge_mt5["rougeL"], rouge_mbart["rougeL"], "ROUGE-L"), (bs_mt5["F1"], bs_mbart["F1"], "mBERTScore-F1")]
for ax, (a, b, t) in zip(axes, pairs):
    ax.boxplot([a, b], tick_labels=["mT5", "mBART"]); ax.set_title(t)
plt.suptitle("Per-article score distribution"); plt.tight_layout(); plt.show()

In [ ]:
# Head-to-head: on how many articles does each model obtain the higher ROUGE-L / mBERTScore?
h2h = pd.DataFrame({
    "mT5 better": [(rouge_mt5["rougeL"] > rouge_mbart["rougeL"]).sum(), (bs_mt5["F1"] > bs_mbart["F1"]).sum()],
    "mBART better": [(rouge_mbart["rougeL"] > rouge_mt5["rougeL"]).sum(), (bs_mbart["F1"] > bs_mt5["F1"]).sum()],
    "tie": [(rouge_mbart["rougeL"] == rouge_mt5["rougeL"]).sum(), (bs_mbart["F1"] == bs_mt5["F1"]).sum()],
}, index=["ROUGE-L", "mBERTScore-F1"])
h2h

In [ ]:
# Save generated summaries for inspection
out = pd.DataFrame({"article": articles, "reference": references,
                    "mT5_summary": mt5_summaries, "mBART_summary": mbart_summaries,
                    "mT5_rougeL": rouge_mt5["rougeL"].round(4), "mBART_rougeL": rouge_mbart["rougeL"].round(4),
                    "mT5_mBERTScore": bs_mt5["F1"].round(4), "mBART_mBERTScore": bs_mbart["F1"].round(4)})
out.to_csv("hindi_xlsum_generated_summaries.csv", index=False, encoding="utf-8-sig")
out[["reference", "mT5_summary", "mBART_summary"]].head(5)